In [1]:
##### Calculates final capital and labor intensities using final production and capital/labor rasters (after re-scaling)

import os
import pandas as pd
import geopandas as gpd
import rioxarray as rio
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from glob import glob
import rasterio
from rasterio.warp import reproject, Resampling
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
from pyproj import Transformer
from pathlib import Path

#### Intensities (m2 projection)

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# Import data
capital = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD.tif")
capital_p10 = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD_p10.tif")
capital_p90 = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD_p90.tif")

labor = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_jobs.tif")
labor_p10 = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_jobs_p10.tif")
labor_p90 = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_jobs_p90.tif")

production = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/total_production_tonnes_2020.tif")

In [3]:
##### Zero out small capital/labor values before computing intensities
# avoids extremely small intensities, but preserves NaN as NaN

crs = capital.rio.crs 

def zero_small_values(da, threshold=1):
    return xr.where(
        da.isnull(),      # if original is NaN, keep NaN
        da,
        xr.where(da >= threshold, da, 0)  # otherwise zero out small values
    )

capital = zero_small_values(capital)
capital_p10 = zero_small_values(capital_p10)
capital_p90 = zero_small_values(capital_p90)

labor = zero_small_values(labor)
labor_p10 = zero_small_values(labor_p10)
labor_p90 = zero_small_values(labor_p90)

In [4]:
##### Calculate and save intensity rasters (central, p10, p90)
def compute_and_save_intensity(numerator, production_masked, crs, scale, out_path):
    # Explicitly mask out any case where production is invalid (NaN, incl. <0.1 already masked)
    # or where the numerator (capital/labor) itself is NaN
    invalid = production_masked.isnull() | numerator.isnull()

    intensity = (numerator / production_masked) * scale
    intensity = intensity.where(~invalid)          # enforce NaN where inputs are invalid
    intensity = intensity.where(np.isfinite(intensity))  # catch any remaining inf/-inf

    intensity = intensity.rio.write_nodata(np.nan)
    intensity = intensity.rio.write_crs(crs)
    intensity.rio.to_raster(out_path, dtype="float32", compress="LZW")
    return intensity

# mask production to get rid of very small production blow up in intensities 
production_masked = production.where(production >= 0.1)

out_dir = f"{cd}/Results/Raster_model"

variants = {
    "": {"capital": capital, "labor": labor},
    "_p10": {"capital": capital_p10, "labor": labor_p10},
    "_p90": {"capital": capital_p90, "labor": labor_p90},
}

results = {}

for suffix, data in variants.items():
    results[f"capital{suffix}"] = compute_and_save_intensity(
        numerator=data["capital"],
        production_masked=production_masked,
        crs=crs,
        scale=1,
        out_path=f"{out_dir}/reprojected/capital_intensity_USD_per_tonne{suffix}.tif",
    )
    results[f"labor{suffix}"] = compute_and_save_intensity(
        numerator=data["labor"],
        production_masked=production_masked,
        crs=crs,
        scale=1,
        out_path=f"{out_dir}/reprojected/labor_intensity_jobs_per_tonne{suffix}.tif",
    )

#### Intensities (4326 projection)

In [5]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# Import data
capital = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_capital_USD.tif")
capital_p10 = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_capital_USD_p10.tif")
capital_p90 = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_capital_USD_p90.tif")

labor = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_jobs.tif")
labor_p10 = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_jobs_p10.tif")
labor_p90 = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_jobs_p90.tif")

production = rio.open_rasterio(f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif")

In [6]:
##### Zero out small capital/labor values before computing intensities
# avoids extremely small intensities, but preserves NaN as NaN

crs = capital.rio.crs 

def zero_small_values(da, threshold=1):
    return xr.where(
        da.isnull(),      # if original is NaN, keep NaN
        da,
        xr.where(da >= threshold, da, 0)  # otherwise zero out small values
    )

capital = zero_small_values(capital)
capital_p10 = zero_small_values(capital_p10)
capital_p90 = zero_small_values(capital_p90)

labor = zero_small_values(labor)
labor_p10 = zero_small_values(labor_p10)
labor_p90 = zero_small_values(labor_p90)

In [7]:
##### Calculate and save intensity rasters (central, p10, p90)
def compute_and_save_intensity(numerator, production_masked, crs, scale, out_path):
    # Explicitly mask out any case where production is invalid (NaN, incl. <0.1 already masked)
    # or where the numerator (capital/labor) itself is NaN
    invalid = production_masked.isnull() | numerator.isnull()

    intensity = (numerator / production_masked) * scale
    intensity = intensity.where(~invalid)          # enforce NaN where inputs are invalid
    intensity = intensity.where(np.isfinite(intensity))  # catch any remaining inf/-inf

    intensity = intensity.rio.write_nodata(np.nan)
    intensity = intensity.rio.write_crs(crs)
    intensity.rio.to_raster(out_path, dtype="float32", compress="LZW")
    return intensity

# mask production to get rid of very small production blow up in intensities 
production_masked = production.where(production >= 0.1)

out_dir = f"{cd}/Results/Raster_model"

variants = {
    "": {"capital": capital, "labor": labor},
    "_p10": {"capital": capital_p10, "labor": labor_p10},
    "_p90": {"capital": capital_p90, "labor": labor_p90},
}

results = {}

for suffix, data in variants.items():
    results[f"capital{suffix}"] = compute_and_save_intensity(
        numerator=data["capital"],
        production_masked=production_masked,
        crs=crs,
        scale=1,
        out_path=f"{out_dir}/capital_intensity_USD_per_tonne{suffix}.tif",
    )
    results[f"labor{suffix}"] = compute_and_save_intensity(
        numerator=data["labor"],
        production_masked=production_masked,
        crs=crs,
        scale=1,
        out_path=f"{out_dir}/labor_intensity_jobs_per_tonne{suffix}.tif",
    )